# Entrenar "oye matix" (openWakeWord, espanol)

Genera el modelo de wake word **"oye matix"** y exporta `oye_matix.onnx` compatible con Matix (cadena mel -> embedding -> clasificador; entrada `[1,16,96]`, salida `[1,1]`).

**Este notebook ya trae TODOS los arreglos** que hicieron falta (openWakeWord es viejo y Colab trae todo nuevo, asi que hay que parchear varias incompatibilidades). Solo segui los pasos.

**ANTES DE CORRER:**
1. Entorno de ejecucion -> Cambiar tipo de entorno -> **GPU (T4)** -> Guardar.
2. Manten la **laptop enchufada y que NO se suspenda**, y la pestana abierta hasta el final. Si se apaga, Colab se desconecta y se pierde todo.
3. Entorno de ejecucion -> **Ejecutar todo**. Corre solo (~30-40 min). Las celdas de entrenar se ponen **rojas** si fallan (no te fies del check verde de las de `!comando`: por eso las pasamos a Python).

El modelo final se **guarda en tu Google Drive** para que no se pierda.

In [ ]:
!nvidia-smi

## 1. Instalar openWakeWord + Piper + extras

In [ ]:
# Clona openWakeWord e instala TODO lo necesario (incluye los paquetes que faltaban:
# pronouncing, onnx). Tarda ~3-4 min. Las advertencias de pip (transformers/fsspec/
# huggingface-hub) son RUIDO, no rompen nada.
!git clone https://github.com/dscripka/openWakeWord
%pip install -q -e ./openWakeWord
%pip install -q piper-tts pronouncing onnx
%pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
  speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
  acoustics==0.2.6 'datasets[audio]==2.14.6'
print('instalacion lista')

## 2. Arreglos de compatibilidad + modelos base

Parchea torch-audiomentations, crea el dummy generate_samples y descarga los modelos base de openWakeWord. Todo lo que descubrimos que faltaba, junto.

In [ ]:
# Arreglos de compatibilidad (todos juntos). Necesarios para que train.py corra.
import glob, re, os, sys

# a) torch-audiomentations llama torchaudio.set_audio_backend() AL IMPORTARSE, y
#    torchaudio nuevo la elimino. Editamos el archivo SIN importar el paquete.
for f in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    s = open(f).read()
    open(f, 'w').write(re.sub(r'torchaudio\.set_audio_backend\([^)]*\)', 'pass', s))
print('a) torch-audiomentations parcheado')

# b) train.py importa 'generate_samples' (de piper-sample-generator) aunque solo se
#    use con --generate_clips (que NO usamos). Creamos un modulo vacio.
d = '/content/openWakeWord/piper-sample-generator'
os.makedirs(d, exist_ok=True)
open(d + '/generate_samples.py', 'w').write(
    'def generate_samples(*a, **k):\n    raise RuntimeError("no se usa en modo augment")\n')
print('b) dummy generate_samples creado')

# c) Modelos base de openWakeWord (melspectrograma + embedding): con pip no vienen.
sys.path.insert(0, '/content/openWakeWord')   # para poder importar openwakeword aqui
import openwakeword.utils
openwakeword.utils.download_models()
print('c) modelos base:', len(glob.glob('/content/openWakeWord/openwakeword/resources/models/*.onnx')), '.onnx')

## 3. Voces de Piper en espanol

In [ ]:
# 8 voces de Piper en espanol (es_AR, es_ES x5, es_MX x2).
import os, urllib.request
BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/es'
voices = [('es_AR','daniela','high'), ('es_ES','carlfm','x_low'), ('es_ES','davefx','medium'),
          ('es_ES','sharvard','medium'), ('es_ES','mls_9972','low'), ('es_ES','mls_10246','low'),
          ('es_MX','ald','medium'), ('es_MX','claude','high')]
os.makedirs('voices', exist_ok=True)
for loc, name, q in voices:
    stem = f'{loc}-{name}-{q}'
    for ext in ('.onnx', '.onnx.json'):
        try:
            urllib.request.urlretrieve(f'{BASE}/{loc}/{name}/{q}/{stem}{ext}', f'voices/{stem}{ext}')
        except Exception as e:
            print('FALLO', stem + ext, e)
print('voces:', len([v for v in os.listdir('voices') if v.endswith('.onnx')]))

## 4. Sintetizar "oye matix" (~320 train + ~64 test) a 16 kHz

In [ ]:
# Sintetiza 'oye matix' en todas las voces (con variacion) y remuestrea a 16 kHz
# mono (openWakeWord lo exige). ~8 voces * 2 settings * 20 reps = ~320 train.
import os, subprocess, uuid, itertools, glob
ROOT = '/content/my_custom_model/oye_matix'
os.makedirs(ROOT + '/positive_train', exist_ok=True)
os.makedirs(ROOT + '/positive_test', exist_ok=True)
PHRASE = 'oye matix'
length_scales = [0.9, 1.1]; noise = [0.6]
def synth(outdir, reps):
    vs = [v for v in os.listdir('voices') if v.endswith('.onnx')]
    hechos = 0
    for v in vs:
        for ls, ns in itertools.product(length_scales, noise):
            for _ in range(reps):
                tmp = '/content/_t.wav'; fn = os.path.join(outdir, uuid.uuid4().hex + '.wav')
                subprocess.run(['piper', '-m', f'voices/{v}', '-f', tmp,
                                '--length_scale', str(ls), '--noise_scale', str(ns)],
                               input=PHRASE.encode(), check=True,
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', fn],
                               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                hechos += 1
                if hechos % 50 == 0: print('  ...', hechos, 'clips')
    print(outdir, '->', hechos)
synth(ROOT + '/positive_train', reps=20)
synth(ROOT + '/positive_test',  reps=4)
print('TOTAL train:', len(glob.glob(ROOT + '/positive_train/*.wav')),
      '| test:', len(glob.glob(ROOT + '/positive_test/*.wav')))

## 5. Negativos + RIR + features precomputadas

In [ ]:
# Negativos: RIR (reverb) + ~2000 h de features precomputadas + features de validacion.
import os, glob, shutil
from huggingface_hub import hf_hub_download, snapshot_download
os.makedirs('mit_rirs', exist_ok=True)
snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset', local_dir='mit_rirs')
# Aplanar los RIR .wav: vienen en subcarpeta y openWakeWord le pasaria la carpeta a
# torchaudio.load (que falla). Los copiamos planos a /content/rirs_flat.
os.makedirs('/content/rirs_flat', exist_ok=True)
for w in glob.glob('/content/mit_rirs/**/*.wav', recursive=True):
    shutil.copy(w, '/content/rirs_flat/' + os.path.basename(w))
print('RIR wavs:', len(glob.glob('/content/rirs_flat/*.wav')))
hf_hub_download('davidscripka/openwakeword_features', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
               repo_type='dataset', local_dir='/content')
hf_hub_download('davidscripka/openwakeword_features', 'validation_set_features.npy',
               repo_type='dataset', local_dir='/content')
print('negativos listos')

## 6. Config YAML

In [ ]:
cfg = '''
target_phrase: ["oye matix"]
model_name: "oye_matix"
output_dir: "/content/my_custom_model"
piper_sample_generator_path: "./piper-sample-generator"
n_samples: 320
n_samples_val: 64
tts_batch_size: 50
rir_paths: ["/content/rirs_flat"]
background_paths: []
background_paths_duplication_rate: []
custom_negative_phrases: []
augmentation_rounds: 1
augmentation_batch_size: 16
batch_n_per_class: 128
feature_data_files:
  ACAV100M_sample: "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
false_positive_validation_data_path: "/content/validation_set_features.npy"
model_type: "dnn"
layer_size: 32
steps: 20000
max_negative_weight: 1500
target_accuracy: 0.6
target_recall: 0.25
target_false_positives_per_hour: 0.2
'''
open('/content/oye_matix.yaml', 'w').write(cfg)
print('config escrita')

### Generar clips NEGATIVOS (otras frases en espanol)

In [ ]:
# Negativos en audio: openWakeWord exige clips en negative_train/test (no acepta
# vacio). Generamos OTRAS frases en espanol (que NO son 'oye matix') con las mismas
# voces, para que el modelo aprenda a distinguir. (Ademas estan las features
# precomputadas de 2000 h que ya bajamos.)
import os, subprocess, uuid, glob, random
ROOT = '/content/my_custom_model/oye_matix'
os.makedirs(ROOT+'/negative_train', exist_ok=True)
os.makedirs(ROOT+'/negative_test', exist_ok=True)
NEGS = ['hola que tal','buenos dias','como estas','que hora es','abre la puerta','oye amigo',
        'matias ven aca','hey jarvis','reproduce musica','enciende la luz','muchas gracias',
        'oye que pasa','dime la hora','escucha esto','vamos a comer','donde estas','apaga todo',
        'sube el volumen','llama a mama','cuanto cuesta','que tiempo hace','no entiendo nada']
vs = [v for v in os.listdir('voices') if v.endswith('.onnx')]
def gen(outdir, reps):
    for i in range(reps):
        ph = random.choice(NEGS); v = random.choice(vs); tmp = '/content/_n.wav'
        fn = os.path.join(outdir, uuid.uuid4().hex + '.wav')
        subprocess.run(['piper','-m',f'voices/{v}','-f',tmp,
                        '--length_scale',str(random.choice([0.9,1.0,1.1])),'--noise_scale','0.6'],
                       input=ph.encode(), check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(['ffmpeg','-y','-i',tmp,'-ar','16000','-ac','1',fn],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if (i+1) % 40 == 0: print('  ...', i+1)
print('generando negativos (~4-6 min)...')
gen(ROOT+'/negative_train', 150); gen(ROOT+'/negative_test', 30)
print('neg train:', len(glob.glob(ROOT+'/negative_train/*.wav')),
      '| neg test:', len(glob.glob(ROOT+'/negative_test/*.wav')))

## 7. Aumentar + features

In [ ]:
# subprocess CAPTURANDO la salida (sin capturar, el error de train.py no se ve en
# la celda). Si returncode != 0, la celda se pone roja y ademas se ve el error.
import subprocess
print('Aumentando (unos minutos)...')
r = subprocess.run('cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --augment_clips', shell=True, capture_output=True, text=True)
print(r.stdout[-3000:])
print('===== ERROR (STDERR) =====')
print(r.stderr[-3000:])
assert r.returncode == 0, 'El AUGMENT fallo: mira el STDERR de arriba.'
print('AUGMENT OK')

## 8. Entrenar el clasificador

In [ ]:
import subprocess, glob
print('Entrenando...')
r = subprocess.run('cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --train_model', shell=True, capture_output=True, text=True)
print(r.stdout[-3000:])
print('===== ERROR (STDERR) =====')
print(r.stderr[-3000:])
assert r.returncode == 0, 'El TRAIN fallo: mira el STDERR de arriba.'
ms = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)
assert ms, 'Entreno pero no encuentro oye_matix.onnx'
print('TRAIN OK ->', ms[0])

## 9. Verificar interfaz ONNX (entrada [1,16,96] f32, salida [1,1])

In [ ]:
import onnx, glob
p = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
m = onnx.load(p)
print('archivo:', p)
print('inputs :', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print('outputs:', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])
print('Debe ser entrada [1, 16, 96] y salida [1, 1].')

## 10. Guardar en Google Drive + descargar

Acepta el permiso para montar tu Drive. Queda en **Mi unidad / oye_matix.onnx**.

In [ ]:
import glob, shutil
p = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy(p, '/content/drive/MyDrive/oye_matix.onnx')
    print('GUARDADO en tu Google Drive (Mi unidad) como oye_matix.onnx')
except Exception as e:
    print('No pude montar Drive:', e)
from google.colab import files
files.download(p)

## 11. Integrarlo en Matix

Pone `oye_matix.onnx` en `MATIX/oye_matix.onnx` en la PC y avisame: yo verifico, hago el swap en ambos pipelines, build y publico el APK nuevo.